# 11. 상수 스윕 — Colab

챔피언 스택(`wonwoo`)에서 **한 축씩** 바꿔 제출 zip을 찍는다. 라운드마다 3번 셀만 고친다.

## 왜 이 방식인가

`reports/10`에서 확인한 것: **혼합·가중 상수의 LB 응답은 포물선이다.**
Brier가 2차 손실이고 예측이 좁은 띠(0.32~0.66)에 몰려 `σ'(z)=p(1−p)`가 거의 일정하기 때문이다.
α 축에서 5점으로 세운 2차식이 측정 범위 밖 0.145 외삽을 **0.0003 오차**로 맞혔다.

현 챔피언이 항상 한 점을 주므로 **축당 2슬롯이면 곡선이 확정된다.**

## 다룰 수 있는 축 세 종류

| 키 | 대상 | 경로 | 재학습 |
| --- | --- | --- | :-: |
| `cfg` | `CTX_AXES` β · `ASOF_AXES` β · `CTX_K` · `CTX_HALF_LIFE` · `ASOF_STRENGTH` | `config` 주입 후 `--artifacts-only` | 불필요 |
| `_vw` | MLP 변형 비중 (`emb4` : `noid`) | `artifacts.joblib` 의 `mlp_weights` 패치 | 불필요 |
| `_c` | **확률공간 스케일** | `predict.py` 3줄 삽입 | 불필요 |

`_vw`가 별도 경로인 이유: `train.py:380`이 `old["mlp_weights"]`를 읽으므로
`config.MLP_VARIANT_WEIGHT`를 바꿔도 `--artifacts-only`에서 **조용히 무시된다.**

### `_c` — 확률공간 스케일의 근거

`y ∈ {0,1}`이라 `y² = y`이고, Brier는 항등식으로 분해된다:

```
BS = Var(p) − 2·Cov(p,y) + (p̄ − r)² + r(1−r)
```

`p' = P0 + c·(p − P0)` 로 아핀 변환하면 `Var → c²Var`, `Cov → c·Cov` 이므로

```
목적함수 = 2c·Cov − c²·Var       →   c* = Cov(p,y)/Var(p)
이득     = 1e5 · Var(p) · (c*−1)² / r(1−r)
```

**c에 대해 정확히 2차다** — α 축과 달리 근사가 없다.
원우 §7.19는 **로짓 공간**(`z' = c·z`)으로 재서 두 fold가 반대를 가리켜 기각됐다.
확률 공간이 나은 점: 곡선이 정확히 2차 · 평균 이동이 P0 기준으로만 일어나 `delta` 재해결 불필요 ·
`train.py`를 안 건드림.

`PROB_CENTER`와 `c`는 **동결 스칼라 리터럴**로 파일에 박힌다.
배치 평균을 쓰면 각 행이 다른 행에 의존해 **대회 규정(행 독립 예측) 위반**이다.

## 열지 않는 축

**`TARGET_PRIOR_2025`.** 레벨 손실은 예측평균 = 실제평균일 때 최소이므로 LB 곡선의 꼭짓점이
곧 **평가 데이터 라벨의 평균값**이다. 그것을 측정해 전 행의 예측을 옮기는 것은 규칙이 금지하는
*"전체 평가 데이터의 분포를 이용한 보정"*이다 (원우 규칙 2).
학습 데이터만으로 재추정하는 것은 정당하나, 2026-08-21에 시도해 **기각**했다
(시즌말 기준 예측기가 시즌평균 기준보다 나쁨: MAE 0.0155 vs 0.0120).

## 준비물

```
MyDrive/LG_aimers_9th/
├── data/                 train.csv 등
└── champion_src.tar.gz   원우 최신 커밋에서 재생성한 것 (노트북 10 상단에 절차)
```

런타임: **CPU · 고용량 RAM** (8 vCPU). GPU 아님 — torch 휠이 달라진다.

## 소요

`FULL_TRAIN=True` 약 40분 + 축당 약 5분. `FULL_TRAIN=False` 면 축당 약 5분.

## 1. 환경

In [ ]:
import sys, platform, subprocess
print('Python  ', sys.version.split()[0])
print('Platform', platform.machine(), platform.system(), '· vCPU', __import__('os').cpu_count())
assert platform.machine() == 'x86_64', 'x86_64가 아니다 — 재현 충실도가 떨어진다'

PINS = ['lightgbm==4.6.0', 'numpy==2.2.6', 'pandas==2.3.3', 'scipy==1.15.2',
        'joblib==1.4.2', 'scikit-learn==1.6.1', 'pyarrow==19.0.1']
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *PINS], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch==2.13.0+cpu',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
print('설치 완료')

In [ ]:
# 불일치가 있으면 [런타임] -> [세션 다시 시작] 후 이 셀부터 다시 실행
import importlib.metadata as md
EXPECT = {'lightgbm':'4.6.0','numpy':'2.2.6','pandas':'2.3.3','scipy':'1.15.2',
          'joblib':'1.4.2','scikit-learn':'1.6.1','pyarrow':'19.0.1'}
bad = [(p, w, md.version(p)) for p, w in EXPECT.items() if md.version(p) != w]
for p, w in EXPECT.items():
    print(f"{'ok ' if md.version(p)==w else 'X  '}{p:16s} {md.version(p):12s} (기대 {w})")
print(f"{'ok ' if md.version('torch').startswith('2.13.0') else 'X  '}{'torch':16s} {md.version('torch')}")
assert not bad, f'버전 불일치: {bad} — 런타임 재시작 후 다시 실행'

## 2. 데이터 · 코드 · 모델 복구

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, shutil, os, tarfile, zipfile, json, time, textwrap
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/LG_aimers_9th')
ROOT  = Path('/content/champion')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)

with tarfile.open(DRIVE / 'champion_src.tar.gz') as tf:
    tf.extractall(ROOT)
assert (ROOT/'src'/'train.py').is_file() and (ROOT/'submit'/'script.py').is_file()
print('코드 전개:', len(list((ROOT/'src').glob('*.py'))), '개 .py')

for name in ['train.csv', 'test.csv', 'sample_submission.csv']:
    src, dst = DRIVE/'data'/name, ROOT/'data'/name
    assert src.is_file(), f'{src} 없음'
    if not dst.exists():
        os.symlink(src, dst)

h = hashlib.sha256()
with open(ROOT/'data'/'train.csv', 'rb') as f:
    for blk in iter(lambda: f.read(1 << 20), b''):
        h.update(blk)
assert h.hexdigest() == 'd2081186b458b49f60b082be480c273135833e15ba59a76d033af28bcf8763ff', \
    'train.csv 가 기록된 파일과 다르다'
print('train.csv 무결성 확인')

In [ ]:
# ── 베이스 확보 ────────────────────────────────────────────────────
# FULL_TRAIN = True   원우 스택이 바뀌었을 때. 약 35분. 모델 31개
# FULL_TRAIN = False  이미 만든 베이스를 zip 에서 복구. 약 1분
#
# 2026-08-22: 원우가 init_score(§7.111, +5.30)와 F전문가(§7.101, +2.27)를 넣어
#             1084.14 가 됐다. fspec 은 **새 모델**이라 --artifacts-only 로는 못 만든다
#             (train.py 가 명시적으로 SystemExit 한다). 그래서 전체 학습이 필요하다.
FULL_TRAIN = True
BASE = ROOT / 'models' / 'base'

import time
if FULL_TRAIN:
    (ROOT / 'trainer.py').write_text(textwrap.dedent('''
        import os, sys
        from pathlib import Path
        ROOT = Path('/content/champion')
        out = ROOT / 'models' / 'base'
        out.mkdir(parents=True, exist_ok=True)
        os.environ['LGA_MODEL_DIR'] = str(out)
        sys.path.insert(0, str(ROOT / 'src')); os.chdir(ROOT)
        import config
        print(f'USE_INIT_SCORE={config.USE_INIT_SCORE}  FSPEC_ALPHA={config.FSPEC_ALPHA}'
              f'  MLP_ALPHA={config.MLP_ALPHA}', flush=True)
        sys.argv = ['train.py']
        import train
        train.main()
    '''), encoding='utf-8')

    t0 = time.time()
    p = subprocess.Popen([sys.executable, '-u', str(ROOT/'trainer.py')], cwd=ROOT,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    log = []
    for line in p.stdout:
        print(line, end='', flush=True)
        log.append(line)
    if p.wait():
        raise SystemExit(f'학습 실패 (종료코드 {p.returncode})')
    log = ''.join(log)
    print(f'\n총 {time.time()-t0:.0f}초')

    for probe, what in [('셀 1,571개', 'δ hand'), ('셀 3,105개', 'δ count4'),
                        ('71,199행', 'soft label'),
                        ('USE_INIT_SCORE=True', 'init_score 켜짐')]:
        print(('ok ' if probe in log else 'X  ') + f'{what}: {probe}')
else:
    src_zip = DRIVE / 'outputs' / 'alpha_probe' / 'p_base.zip'
    assert src_zip.is_file(), f'{src_zip} 없음 — FULL_TRAIN=True 로 만들 것'
    BASE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_zip) as z:
        for n in z.namelist():
            if n.startswith('model/') and not n.endswith('/'):
                (BASE / Path(n).name).write_bytes(z.read(n))
    print(f'복구 완료: {src_zip.name}')

# ⚠️ F전문가는 `lgbm_fspec_{seed}.txt` 로 저장된다 (train.py:31,
#    predict.py:60 도 같은 이름으로 읽는다). `lgbm_*` 로 뭉뚱그려 세면
#    20+3=23 이 나오고 fspec 은 0 으로 보인다. 접두어로 갈라야 한다.
fspec_files = sorted(BASE.glob('lgbm_fspec_*.txt'))
lgbm_files = [p for p in BASE.glob('lgbm_*.txt') if not p.name.startswith('lgbm_fspec_')]
mlp_files = sorted(BASE.glob('mlp_*.npz'))
meta = json.loads((BASE/'meta.json').read_text())

print(f'\n베이스: {BASE}')
print(f'  lgbm {len(lgbm_files)} · mlp {len(mlp_files)} · fspec {len(fspec_files)}'
      f' · mlp_alpha={meta["mlp_alpha"]} · init={meta["use_init_score"]}')
print(f'  fspec_members(meta) = {meta.get("fspec_members")}')
assert len(lgbm_files) == 20, f'lgbm {len(lgbm_files)}개 (기대 20)'
assert len(mlp_files) == 8, f'mlp {len(mlp_files)}개 (기대 8)'
assert len(fspec_files) == len(meta.get('fspec_members') or []), \
    'fspec 파일 수와 meta 의 fspec_members 가 다르다'
assert len(fspec_files) >= 1, 'fspec 멤버가 없다 — FSPEC_ALPHA>0 인데 F전문가가 안 만들어졌다'
assert (BASE/'artifacts.joblib').is_file()

## 3. 스윕 정의

`SWEEP`의 각 항목이 zip 하나가 된다. **이 셀만 바꾸면 된다** — 아래 셀들은 전부 `SWEEP`을
순회할 뿐이고 태그를 하드코딩한 곳이 없다. (`models/a40`은 스윕 대상이 아니라 **모델 복구원**이다.)

### 2026-08-21 2차 — MLP 변형 비중 (`noid` 몫)

α 축은 8점으로 종결됐다 (`α* = 0.74499`, 신기록 **1076.0280976273**).
다음은 **MLP 내부의 변형 비중**이다. α와 같은 성질의 질문 — 예측기 간 상대 신뢰도다.

**왜 이 축인가.** 배포값은 `emb4 0.625 : noid 0.375`인데 `wm = vw/tot/nseed`로 펼치면
`emb4 0.625/5 = 0.125`, `noid 0.375/3 = 0.125`로 **8멤버 균등 평균**이다.
그런데 §7.24가 이렇게 기록했다:

> `noid`는 핸디캡 멤버가 아니라 **더 센** 멤버였다.

**더 센 멤버가 3/8만 갖고 있다.** §7.27이 비율 변경을 기각했지만 그건 홀드아웃 게이트였고,
α에서 그 게이트가 부호를 틀린 것이 증명됐다.

| `noid` 몫 | 태그 | 멤버당 가중 (emb4 / noid) | LB |
| ---: | --- | --- | ---: |
| **0.375** | (`p_a75`) | 0.1250 / 0.1250 | **1076.0280976273** |
| 0.500 | `n50` | 0.1000 / 0.1667 | 측정 대상 |
| 0.625 | `n62` | 0.0750 / 0.2083 | 측정 대상 |

챔피언이 그대로 기준점이라 **2슬롯으로 3점 곡선**이 나온다.

### 이 축은 config 로 못 바꾼다 — 아티팩트를 직접 고친다

`train.py:380`이 `old["mlp_weights"]`를 읽으므로 `config.MLP_VARIANT_WEIGHT`를 바꿔도
`--artifacts-only`에서는 **조용히 무시된다.** 대신 `artifacts.joblib`의 `mlp_weights`를
직접 패치한다. 이 값은 순수한 선형 가중 벡터이고 세 곳이 같은 것을 읽으므로 일관된다:

```
load_members(MODEL_DIR, old, ...)   ->  z_mlp_ref 가 새 가중으로 계산되고
raw_ref = (1-α)·lgbm + α·z_mlp      ->  delta 가 그 위에서 재해결되고
_finish(..., mlp_wts, ...)          ->  같은 가중이 zip 에 저장된다
```

`MLP_ALPHA`는 두 태그 모두 **0.75 고정** — 한 번에 한 축만 움직인다.

In [ ]:
# 태그 -> 사양. 라운드마다 이 셀만 바꾸고 아래를 다시 실행한다.
#
#   'cfg'  config 속성 오버라이드 (train import 전에 setattr)
#   '_vw'  MLP 변형 비중. config 로는 안 먹어 artifacts.joblib 을 패치한다
#   '_c'   확률공간 스케일. predict.py 에 3줄 삽입. c=1.0 이면 완전 항등
#
# cfg 도 _vw 도 없으면 베이스 모델을 그대로 쓴다 (재적합 없음).

PROB_CENTER = 0.4801     # 동결 스칼라. 245,789행 더미 예측 평균

# ── 2026-08-23 — 재현 판정 + 온도 + noid ─────────────────────────
SWEEP = {
    'base': {},                                            # 원우 s29 = 1084.1432 대조
    'c115': {'_c': 1.15},                                   # 온도. 아래 근거 참조
    'c130': {'_c': 1.30},
    'n12':  {'_vw': {'emb4': 0.875, 'noid': 0.125}},        # noid 아래쪽
    'n00':  {'_vw': {'emb4': 1.000, 'noid': 0.000}},
}

# ── 온도 c 를 1.15 / 1.30 으로 고른 근거 ──────────────────────────
# 원우 experiments/note_bs_decomp_lowfk.md (2026-08-21, fold2024 십분위):
#     REL 0.000040 · RES 0.002053 · 예측 범위 0.341~0.618
#     최하 빈 f_k=.4156 vs ō_k=.4072  |  최상 빈 .5641 vs .5782
#     "양 꼬리 모두 f_k 가 ō_k 보다 안쪽(under-sharp) — 전역 온도 c>1 의 자리"
# 그의 꼬리 수치로 c 를 역산하면 (중심 ȳ_2024 = .4861):
#     최하  (.4072−.4861)/(.4156−.4861) = 1.119
#     최상  (.5782−.4861)/(.5641−.4861) = 1.181      → c ≈ 1.15
# 분산식 역산(Var(p) 대리값 기준)도 같은 대역을 가리킨다. 두 경로가 독립적으로 일치.
#
# ⚠️ 상한이 있다. 단조 변환이라 RES 는 못 건드리고 REL 만 줄인다.
#    REL 0.000040 → BSS 로 약 16점이 상한 언저리다. +20 은 낙관치.
#
# ⚠️ 기각 이력: §7.19(로짓 온도)와 jointbeta 변형 C 가 **fold 간 방향 불일치**로 기각했다
#    (fold2023 은 c<0.90, fold2024 는 c>1). LB 는 fold 가 하나라 그 문제가 없다.
#    그리고 확률공간 스케일은 c 에 대해 **정확히 2차**라 3점이면 꼭짓점이 확정된다.

# ── 다음 라운드 후보 ──────────────────────────────────────────────
#   후처리 4축 공통 배율
#     {'b08': {'cfg': {'CTX_AXES': [['hand',1.5],['count4',1.0]],
#                      'ASOF_AXES': [['gap_success',0.024],['gap_bsuccess',0.024]]}},
#      'b12': {'cfg': {'CTX_AXES': [['hand',2.25],['count4',1.5]],
#                      'ASOF_AXES': [['gap_success',0.036],['gap_bsuccess',0.036]]}}}
#   CTX_K 200 -> 100 / 400   ·   ASOF_STRENGTH 35 -> 20 / 60
#
# TARGET_PRIOR_2025 는 열지 않는다 (규정 — 노트북 상단).

for tag, spec in SWEEP.items():
    assert len(f'p_{tag}.zip') <= 30, f'{tag}: 파일명 30자 초과'
    assert set(spec) <= {'cfg', '_vw', '_c'}, f'{tag}: 알 수 없는 키 {set(spec)}'
    c = spec.get('_c', 1.0)
    assert 0.5 <= c <= 2.0, f'{tag}: c={c} 가 상식 범위 밖'
    vw = spec.get('_vw')
    if vw:
        assert abs(sum(vw.values()) - 1.0) < 1e-9, f'{tag}: _vw 합이 1이 아니다'
    print(f'p_{tag}.zip   cfg={spec.get("cfg") or "-"}  vw={vw or "-"}  c={c}')
print(f'\n{len(SWEEP)}개 — 오늘 슬롯 5개에 맞춘다')

In [ ]:
RUNNER = ROOT / 'run_sweep.py'
RUNNER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', required=True)
    ap.add_argument('--reuse', required=True)
    ap.add_argument('--overrides', required=True)   # JSON (config 속성)
    ap.add_argument('--mlp-vw', default=None)       # JSON (변형 -> 비중)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    out = ROOT / a.out
    if out.exists(): shutil.rmtree(out)
    shutil.copytree(ROOT / a.reuse, out)      # 베이스를 보존한다
    os.environ['LGA_MODEL_DIR'] = str(out)
    sys.path.insert(0, str(ROOT / 'src'))
    os.chdir(ROOT)

    # ── MLP 변형 비중은 config 로 못 바꾼다 (train.py:380 이 old["mlp_weights"] 를
    #    읽는다). artifacts.joblib 을 직접 고친다. 순수 선형 가중 벡터라 안전하고
    #    load_members / _finish / predict.py 가 모두 같은 값을 읽어 일관된다.
    if a.mlp_vw:
        import joblib
        vw = json.loads(a.mlp_vw)
        b = joblib.load(out / 'artifacts.joblib')
        tags, cur = b['mlp_members'], list(b['mlp_weights'])
        assert max(cur) - min(cur) < 1e-9, f'베이스 가중이 균등이 아니다: {cur}'
        assert abs(sum(cur) - 1.0) < 1e-9
        variant = [t.rsplit('_', 1)[0] for t in tags]
        assert set(variant) == set(vw), f'변형 불일치 {set(variant)} vs {set(vw)}'
        cnt = {v: variant.count(v) for v in vw}
        new = [vw[v] / sum(vw.values()) / cnt[v] for v in variant]
        assert abs(sum(new) - 1.0) < 1e-9
        print('mlp_weights 패치:', flush=True)
        for t, o, n in zip(tags, cur, new):
            print(f'    {t:18s} {o:.6f} -> {n:.6f}', flush=True)
        b['mlp_weights'] = new
        joblib.dump(b, out / 'artifacts.joblib', compress=3)

    # train 을 import 하기 **전에** config 를 고친다.
    # train.py 는 `from config import (...)` = import 시점의 값 복사이므로,
    # 순서가 뒤바뀌면 조용히 배포값으로 돌아간다.
    import config
    ov = json.loads(a.overrides)
    for k, v in ov.items():
        assert hasattr(config, k), f'config 에 {k} 가 없다'
        if isinstance(v, list):
            v = [tuple(x) if isinstance(x, list) else x for x in v]
        print(f'주입 {k}: {getattr(config, k)} -> {v}', flush=True)
        setattr(config, k, v)

    sys.argv = ['train.py', '--artifacts-only']
    import train
    for k, v in ov.items():                   # 침묵 실패 방어선
        if hasattr(train, k):
            got = getattr(train, k)
            exp = [tuple(x) if isinstance(x, list) else x for x in v] if isinstance(v, list) else v
            assert got == exp, f'주입 실패 train.{k}={got} != {exp}'
    train.main()

    if a.mlp_vw:
        import joblib
        chk = joblib.load(out / 'artifacts.joblib')['mlp_weights']
        assert max(chk) - min(chk) > 1e-9, '저장 후 가중이 다시 균등이다 — 패치가 덮였다'
        print(f'저장 확인: mlp_weights {[round(x,6) for x in chk]}', flush=True)
'''), encoding='utf-8')


def stream(cmd, cwd=ROOT):
    """출력을 한 줄씩 즉시 흘린다. capture_output 은 진행 상황을 가린다."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise SystemExit(f'실패 (종료코드 {p.returncode})')
    return ''.join(lines)


def prepare(tag, spec):
    """이 태그가 쓸 모델 디렉터리를 만들고 경로를 돌려준다.

    cfg 도 _vw 도 없으면 베이스를 그대로 쓴다 — 재적합이 없으므로 베이스와
    비트 단위로 같은 예측이 나온다 (c=1.0 이면 챔피언 재현 판정이 된다).
    """
    cfg, vw = spec.get('cfg') or {}, spec.get('_vw')
    if not cfg and not vw:
        print(f'{tag}: 베이스 그대로 사용 (재적합 없음)')
        return 'models/base'
    cmd = [sys.executable, '-u', str(RUNNER), '--out', f'models/{tag}',
           '--reuse', 'models/base', '--overrides', json.dumps(cfg)]
    if vw:
        cmd += ['--mlp-vw', json.dumps(vw)]
    stream(cmd)
    return f'models/{tag}'


print('러너 작성 완료')

## 4. 스윕 실행 — 태그당 약 45초

`--artifacts-only`이므로 모델은 재사용하고 δ 표 재적합 + prior shift 재해결만 한다.
`logit_delta`가 태그마다 **다른 값**이어야 한다 — 같으면 주입이 안 먹은 것이다.

In [ ]:
MODELS, metas = {}, {}
for tag, spec in SWEEP.items():
    print(f'\n{"="*22} {tag} {"="*22}')
    MODELS[tag] = prepare(tag, spec)
    metas[tag] = json.loads((ROOT / MODELS[tag] / 'meta.json').read_text())

print('\n' + '='*54)
for tag in SWEEP:
    m = metas[tag]
    print(f'  {tag:6s} {MODELS[tag]:14s} logit_delta={m["logit_delta"]:+.5f}  '
          f'init={m["use_init_score"]}  fspec={len(m.get("fspec_members") or [])}  '
          f'c={SWEEP[tag].get("_c", 1.0)}')

# 재적합이 일어난 태그끼리는 logit_delta 가 달라야 한다 (침묵 실패 방어선).
refit = [t for t, s in SWEEP.items() if (s.get('cfg') or s.get('_vw'))]
if len(refit) > 1:
    vals = [metas[t]['logit_delta'] for t in refit]
    assert len(set(vals)) == len(vals), \
        f'logit_delta 가 겹친다 — 오버라이드가 안 먹었을 가능성이 크다: {dict(zip(refit, vals))}'
    print('  재적합 태그들의 logit_delta 가 전부 다르다 — 주입 반영됨')

# 확률 스케일만 다른 태그들은 모델이 같아야 한다 (스케일은 predict.py 에서만 걸린다).
scaled = [t for t, s in SWEEP.items() if s.get('_c', 1.0) != 1.0 and not (s.get('cfg') or s.get('_vw'))]
if scaled:
    assert all(MODELS[t] == 'models/base' for t in scaled), \
        '스케일 전용 태그가 베이스를 안 쓰고 있다'
    print(f'  스케일 전용 태그 {scaled} 는 베이스 모델을 공유한다 — 차이는 c 뿐이다')

## 5. zip 빌드

원우 `build_submit.py`를 감싼다. 그대로 쓰면 네 가지가 깨진다:
`model_dir` 하드코딩 · `submit/` 전체를 훑는 zip · 원우 추적 파일 기록 · s25.. 자동 번호.

In [ ]:
BUILDER = ROOT / 'build_one.py'
BUILDER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, zipfile, ast, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True)
    ap.add_argument('--name', required=True)
    ap.add_argument('--scale', type=float, default=1.0)
    ap.add_argument('--center', type=float, default=0.4801)
    a = ap.parse_args()
    NL = chr(10)          # 이스케이프를 쓰지 않는다 (중첩 문자열에서 겹치기 쉽다)

    ROOT = Path('/content/champion')
    zip_path = ROOT / f'{a.name}.zip'
    assert len(zip_path.name) <= 30, f'파일명 {len(zip_path.name)}자 > 30자'
    if zip_path.exists(): zip_path.unlink()

    staged = ROOT / 'submit' / 'model'
    if staged.exists(): shutil.rmtree(staged)
    shutil.copytree(ROOT / a.models, staged)
    meta = json.loads((staged/'meta.json').read_text())

    strays = [p for p in (ROOT/'submit').iterdir()
              if p.is_dir() and p.name.startswith('model') and p.name != 'model']
    assert not strays, f'submit/ 에 다른 모델이 있다 (zip 이 통째로 삼킨다): {strays}'

    sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
    import build_submit

    # ── 확률공간 스케일 (hyejin 패치) ─────────────────────────────────
    # BS = Var(p) − 2Cov(p,y) + (p̄−r)² + r(1−r) 에서 p' = P0 + c(p−P0) 은
    # Var → c²Var, Cov → c·Cov 이므로 최적 c* = Cov/Var, 곡선은 정확히 2차다.
    # P0·c 는 **동결 스칼라 리터럴**이다 — 배치 평균을 쓰면 행 독립성 위반이다.
    KEY = 'return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)'
    if a.scale != 1.0:
        lines = (ROOT/'src'/'predict.py').read_text(encoding='utf-8').split(NL)
        idx = [i for i, l in enumerate(lines) if l.strip() == KEY]
        assert len(idx) == 1, f'predict.py 에서 삽입 지점을 {len(idx)}곳 찾았다 (1곳이어야)'
        i = idx[0]
        pad = ' ' * (len(lines[i]) - len(lines[i].lstrip()))
        lines[i:i] = [pad + f'PROB_CENTER = {a.center!r}',
                      pad + f'PROB_SCALE = {a.scale!r}',
                      pad + 'p = PROB_CENTER + PROB_SCALE * (p - PROB_CENTER)']
        patched = NL.join(lines)
        ast.parse(patched)
        (ROOT/'submit'/'predict.py').write_text(patched, encoding='utf-8')
        build_submit.COPY_FILES = [f for f in build_submit.COPY_FILES if f != 'predict.py']
        hits = build_submit.leakage_scan([ROOT/'submit'/'predict.py'])
        assert not hits, f'패치본에 금지 호출: {hits}'
        print(f'확률 스케일 패치: c={a.scale} P0={a.center}  (리키지 검사 통과)')

    build_submit.record_history = lambda *x, **k: print('  이력 기록 건너뜀')
    build_submit.log_submission = lambda *x, **k: print('  SUBMISSIONS.md 기록 건너뜀')
    build_submit.next_zip_path  = lambda m, suffix='': zip_path
    build_submit.main()

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        pred = z.read('predict.py').decode('utf-8')
    # F전문가는 lgbm_fspec_{seed}.txt 로 저장된다 — 접두어로 갈라야 한다
    n_f = sum(n.startswith('model/lgbm_fspec_') for n in names)
    n_l = sum(n.startswith('model/lgbm_') for n in names) - n_f
    n_m = sum(n.startswith('model/mlp_')  for n in names)
    assert n_l == 20 and n_m == 8, f'LGBM {n_l} · MLP {n_m} (기대 20 · 8)'
    assert n_f >= 1, 'zip 에 fspec 멤버가 없다'
    if a.scale != 1.0:
        assert f'PROB_SCALE = {a.scale!r}' in pred, 'zip 의 predict.py 에 스케일이 없다'
    else:
        assert 'PROB_SCALE' not in pred, 'c=1.0 인데 스케일 코드가 들어갔다'
    print(f'OK {zip_path.name} alpha={meta["mlp_alpha"]} lgbm={n_l} mlp={n_m} '
          f'fspec={n_f} c={a.scale} {zip_path.stat().st_size/1e6:.1f}MB')
'''), encoding='utf-8')


for tag, spec in SWEEP.items():
    print(f'\n{"="*22} {tag} {"="*22}', flush=True)
    stream([sys.executable, '-u', str(BUILDER),
            '--models', MODELS[tag], '--name', f'p_{tag}',
            '--scale', str(spec.get('_c', 1.0)), '--center', str(PROB_CENTER)])

## 6. 검증 — 압축을 푼 상태에서 실행

`build_submit.py`는 `script.py`를 `submit/` 작업 디렉터리에서 돌리므로
**`submit/`에는 있고 zip에는 없는 파일**을 못 잡는다. 여기서는 빈 임시 디렉터리에 푼다.

행 독립성 임계값 `TOL = 1e-6` — 어제 측정한 잔차는 MLP BLAS 노이즈이며 α에 정비례했다
(2.1e-08/α). 진짜 누출은 1e-3 이상이라 두 자릿수 이상 떨어져 있다.

In [ ]:
import tempfile, pandas as pd, numpy as np
sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
import build_submit as bs

N_ROWS, TOL = 20000, 1e-6
dummy = bs.make_dummy_test(N_ROWS)
HIDDEN = ('__MACOSX', '.DS_Store', 'Thumbs.db', '.ipynb_checkpoints')

def run_script(work, df):
    d = work/'data'; d.mkdir(exist_ok=True)
    df.to_csv(d/'test.csv', index=False)
    pd.DataFrame({'row_id': df['row_id'], 'control_success': .5}).to_csv(
        d/'sample_submission.csv', index=False)
    shutil.rmtree(work/'output', ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, 'script.py'], cwd=work,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:]); print(r.stderr[-3000:]); raise SystemExit('script.py 실패')
    return pd.read_csv(work/'output'/'submission.csv'), time.time()-t0

stats = {}
for tag in SWEEP:
    zp = ROOT/f'p_{tag}.zip'
    print(f'\n{"="*22} {zp.name} {"="*22}')
    assert len(zp.name) <= 30
    with zipfile.ZipFile(zp) as z:
        assert z.testzip() is None, 'CRC 불일치'
        names = z.namelist()
    assert not [n for n in names if any(h in n for h in HIDDEN)], '숨은 파일'
    for req in ['script.py', 'requirements.txt']:
        assert req in names, f'루트에 {req} 없음'
    assert 'script.py' in {n.split('/', 1)[0] for n in names}, '최상위 래퍼 폴더가 있다'
    print(f'  구조 정상 · 구성원 {len(names)}개 · 파일명 {len(zp.name)}자')

    with tempfile.TemporaryDirectory() as tmp:
        work = Path(tmp)/'x'; work.mkdir()
        with zipfile.ZipFile(zp) as z: z.extractall(work)
        assert 'torch' not in (work/'requirements.txt').read_text(), 'requirements 에 torch'

        base, el = run_script(work, dummy)
        proj = el/len(dummy)*245789
        assert list(base.columns) == ['row_id','control_success']
        assert len(base) == len(dummy)
        assert base['row_id'].equals(dummy['row_id'].reset_index(drop=True)), 'row_id 불일치'
        p = base['control_success']
        assert p.notna().all() and p.between(0,1).all()
        assert proj < 480, f'추론 시간 여유 부족 ({proj:.0f}s)'
        stats[tag] = {'mean': float(p.mean()), 'std': float(p.std(ddof=0)),
                      'min': float(p.min()), 'max': float(p.max())}
        print(f'  실행 {el:.1f}s -> 245,789행 환산 {proj:.0f}s / 600s ({proj/6:.0f}%)')
        print(f'  출력 정상  mean={p.mean():.6f}  std={p.std(ddof=0):.6f}  '
              f'[{p.min():.4f}, {p.max():.4f}]')

        ref = base.set_index('row_id')['control_success']
        sh, _ = run_script(work, dummy.sample(frac=1, random_state=42).reset_index(drop=True))
        d1 = (sh.set_index('row_id')['control_success'] - ref).abs().max()
        half = len(dummy)//2
        ch = pd.concat([run_script(work, dummy.iloc[:half].reset_index(drop=True))[0],
                        run_script(work, dummy.iloc[half:].reset_index(drop=True))[0]])
        d2 = (ch.set_index('row_id')['control_success'] - ref).abs().max()
        assert d1 < TOL, f'셔플 후 예측이 달라진다 ({d1:.3e}) — 규정 위반 의심'
        assert d2 < TOL, f'청크 분할 예측이 달라진다 ({d2:.3e}) — 규정 위반 의심'
        print(f'  행 독립성  셔플 {d1:.1e} · 청크 {d2:.1e}'
              + ('' if max(d1,d2) == 0 else '  (BLAS 노이즈)'))

print('\n' + '='*54)
print(f'{len(SWEEP)}개 전부 통과. 제출 가능.')
for t, s in stats.items():
    print(f'   p_{t}.zip   mean={s["mean"]:.6f}  std={s["std"]:.6f}')
print('='*54)

# ── c* 역산기 ──────────────────────────────────────────────────────
# BS = Var(p) − 2Cov(p,y) + (p̄−r)² + r(1−r)  (y∈{0,1} 항등식)
#   => c* = Cov/Var = [BSS·r(1−r)/1e5 + Var(p) + (p̄−r)²] / (2·Var(p))
#   이득 = 1e5 · Var(p) · (c*−1)² / r(1−r)
# ⚠️ Var(p) 는 2024 더미 기준 대리값이다. c* 는 "슬롯을 쓸지" 판단용 추정치이고,
#    확정은 LB 2점(현 챔피언 포함 3점)이 한다 — 곡선이 정확히 2차이므로.
def cstar(bss, std_p, pbar, r=0.480):
    V, u = std_p**2, r*(1-r)
    c = (bss*u/1e5 + V + (pbar-r)**2) / (2*V)
    return c, 1e5*V*(c-1)**2/u

print('\nc* 역산 — 제출 후 아래에 LB 점수를 넣어 실행할 것')
print('예:  for r in (0.475, 0.480, 0.486):')
print('         c, g = cstar(1084.14, stats["base"]["std"], stats["base"]["mean"], r)')
print('         print(f"r={r}  c*={c:.3f}  예상이득={g:+.1f}")')

## 7. Drive 저장

결과는 `hyejin:reports/10_alpha_probe_report.md` §5 표에 이어서 기록한다.

In [ ]:
OUT = DRIVE / 'outputs' / 'alpha_probe'
OUT.mkdir(parents=True, exist_ok=True)
for tag in SWEEP:
    src = ROOT / f'p_{tag}.zip'
    shutil.copy2(src, OUT / src.name)
    print(f'{OUT}/{src.name}   ({src.stat().st_size/1e6:.1f} MB)')

print('\n제출할 것:')
for i, t in enumerate(SWEEP, 1):
    print(f'  {i}. p_{t}.zip')
print('\n비교 기준: p_a75.zip = 1076.0280976273  (α=0.75, 보정축 배율 m=1.0)')
print('결과는 hyejin:reports/10_alpha_probe_report.md 에 기록한다.')